# ===============================================================
# Phase 3: Embed & Index into Chroma
#  - Input : data/processed/chunks.jsonl
#  - Output: vectorstore (persist), collection 'jenosize-ideas'
# ===============================================================

In [1]:
# --- Imports & Config ---
from pathlib import Path
import os, json, math, time
from tqdm import tqdm
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv
load_dotenv()

import chromadb
from chromadb.config import Settings

# 🔁 ใช้ Embedding API (แทน SentenceTransformer)
from app.rag.embeddings_api import embed_docs

# ---- Paths/ENV ----
CHUNKS_FILE = Path(os.getenv("CHUNKS_FILE", "../data/processed/chunks.jsonl"))
CHROMA_DIR  = Path(os.getenv("CHROMA_PERSIST_DIR", "./vectorstore"))
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "jenosize-ideas")

assert CHUNKS_FILE.exists(), f"Not found: {CHUNKS_FILE}"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# ---- Chroma client/collection ----
client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(allow_reset=True))
collection = client.get_or_create_collection(name=COLLECTION_NAME)

print("✅ Ready ->")
print(" - CHUNKS_FILE      :", CHUNKS_FILE.resolve())
print(" - CHROMA_PERSIST   :", CHROMA_DIR.resolve())
print(" - COLLECTION_NAME  :", COLLECTION_NAME)


✅ Ready ->
 - CHUNKS_FILE      : D:\mini-jane-demo\data\processed\chunks.jsonl
 - CHROMA_PERSIST   : D:\mini-jane-demo\notebooks\vectorstore
 - COLLECTION_NAME  : jenosize-ideas


In [2]:
RESET = False  # เปลี่ยนเป็น True ถ้าต้องล้าง

if RESET:
    try:
        client.delete_collection(COLLECTION_NAME)
        collection = client.get_or_create_collection(name=COLLECTION_NAME)
        print("🧹 Reset collection:", COLLECTION_NAME)
    except Exception as e:
        print("⚠️ Reset failed:", e)


In [3]:
# --- Load & Dedupe chunks ---
def make_id(url: str, chunk_index: int) -> str:
    return f"{url}#chunk{int(chunk_index)}"

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    raw = [json.loads(line) for line in f if line.strip()]

# กันซ้ำตาม (url, chunk_index)
seen = set()
chunks = []
for r in raw:
    key = (r.get("url",""), int(r.get("chunk_index", 0)))
    if key in seen:
        continue
    seen.add(key)
    chunks.append({
        "url": r.get("url",""),
        "title": r.get("title",""),
        "category": r.get("category",""),
        "section": r.get("section",""),
        "chunk_index": int(r.get("chunk_index", 0)),
        "text": r.get("text",""),
        "language": r.get("language","en"),
    })

len(chunks), chunks[0] if chunks else None


(446,
 {'url': 'https://www.jenosize.com/en/ideas/experience-the-new-world/event-design-thinking',
  'title': '5 Steps of Event Design Thinking for Memorable Events',
  'category': 'Experience the New World',
  'section': 'Body',
  'chunk_index': 1,
  'text': "5 Key Steps in the Event Design Thinking Process\nEvents are powerful platforms where businesses can unleash creativity, connect with audiences, and leave a lasting impression. But the real question is: how do you design an event that’s truly unforgettable? One proven approach adopted by leading global agencies and organizations is event design thinking—a human-centered design process that puts the event attendee at the core, focusing on emotional engagement, sensory experience, and business goals. What Is Event Design Thinking ? Event design thinking is a strategic planning method that begins with understanding the needs and motivations of event attendees. From there, it moves through a structured process—defining challenges, de

In [4]:
from chromadb.config import Settings
import chromadb, os
from pathlib import Path

# ใช้ค่าจาก .env
from dotenv import load_dotenv
load_dotenv()

CHROMA_DIR = Path(os.getenv("CHROMA_PERSIST_DIR", "./vectorstore"))
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "jenosize-ideas")

client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(allow_reset=True))

# ลบคอลเลกชันเดิม (ถ้ามี)
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"🧹 Deleted old collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"(skip) delete_collection: {e}")

# สร้างใหม่ (จะรับมิติจากการ upsert ครั้งแรก)
collection = client.get_or_create_collection(name=COLLECTION_NAME)
print("✅ Re-created collection:", COLLECTION_NAME, "| count:", collection.count())


🧹 Deleted old collection: jenosize-ideas
✅ Re-created collection: jenosize-ideas | count: 0


In [5]:
# --- Upsert to Chroma with Gemini Embedding API ---
BATCH = 64
added = 0

for i in tqdm(range(0, len(chunks), BATCH), desc="Indexing (Gemini API)"):
    batch = chunks[i:i+BATCH]
    ids   = [make_id(b["url"], b["chunk_index"]) for b in batch]
    docs  = [b["text"] for b in batch]
    metas = [{
        "url": b["url"], "title": b["title"], "category": b["category"],
        "section": b["section"], "chunk_index": b["chunk_index"],
        "language": b.get("language","en")
    } for b in batch]

    # ⬇️ เปลี่ยนจาก embedder.encode(...) → ใช้ Gemini API
    embs = embed_docs(docs)  # -> List[List[float]]

    # ใช้ upsert เพื่อ rerun ได้ไม่ error (ID เดิม = update)
    collection.upsert(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
    added += len(batch)

print(f"✅ Upserted {added} chunks into '{COLLECTION_NAME}'")


Indexing (Gemini API): 100%|██████████| 7/7 [02:50<00:00, 24.29s/it]

✅ Upserted 446 chunks into 'jenosize-ideas'


In [6]:
print("Collection count:", collection.count())


Collection count: 446


In [7]:
# --- Minimal query test (ไม่พึ่ง retriever.py) ---
from app.rag.embeddings_api import embed_query

def quick_search(query: str, k=5):
    q_emb = embed_query(query)
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents","metadatas","distances"]
    )
    hits = []
    for doc, meta, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append((d, meta.get("title",""), meta.get("url",""), meta.get("category","")))
    return hits

for d, title, url, cat in quick_search("AI trends for 2030", k=5):
    print(f"{d:.4f} | {cat} | {title}\n  {url}\n")


0.7546 | Futurist | 9 Experiential Marketing Trends to Watch in 2030
  https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends

0.8070 | Futurist | 9 Experiential Marketing Trends to Watch in 2030
  https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends

0.8479 | Experience the New World | 8 Trending Startup Businesses for Modern Entrepreneurs
  https://www.jenosize.com/en/ideas/experience-the-new-world/top-startup-trends

0.8723 | Futurist | 9 Experiential Marketing Trends to Watch in 2030
  https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends

0.8802 | Futurist | 9 Experiential Marketing Trends to Watch in 2030
  https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends



In [8]:
from app.rag.retriever import get_collection
col = get_collection()
print("count =", col.count())  # ควร > 0


count = 446


In [9]:
from app.rag.retriever import search
hits = search("AI trends for 2030", k=5)
len(hits), hits[0]["meta"] if hits else None


(5,
 {'language': 'en',
  'url': 'https://www.jenosize.com/en/ideas/futurist/experiential-marketing-2030-trends',
  'section': 'Body',
  'category': 'Futurist',
  'title': '9 Experiential Marketing Trends to Watch in 2030',
  'chunk_index': 2})